In [23]:
import os
import sys
import pandas as pd
import sys, os
sys.path.append('..')

import time

# -------------------------------
# NLP
# -------------------------------
from nlp.sentiment_analyzer import SentimentAnalyzer
from nlp.intent_classifier import predict

# -------------------------------
# Retrieval
# -------------------------------
from retrieval.retriever import retrieve
from retrieval.retrieval_optimizer import optimize_retrieval
from retrieval.knowledge_gap import compute_retrieval_confidence

# -------------------------------
# RL + Generator
# -------------------------------
from rl.environment import NexResolveEnv
from generation.generator import ResponseGenerator

# Mock state builder or use data
df = pd.read_csv("../data/final/rl_ready_dataset.csv")
#sample_row = df.iloc[0]
high_sim_rows = df[df["max_sim"] >= 0.55]
sample_row = high_sim_rows.iloc[0]
ticket_text = sample_row["clean_text"]
print(f"Ticket Text: {ticket_text[:200]}...")

Ticket Text: error: error during execution type: bug error: error during execution extension version: 0.39.2 vs code version: code 1.111.0 ( [hash] , 2026-03-06t23:06:10z) os version: windows nt x64 10.0.26200 mod...


## 1. NLP Processing

In [24]:
import time
import os

start = time.time()

# -------------------------------
# SENTIMENT
# -------------------------------
sentiment_model = SentimentAnalyzer()

score = sentiment_model.analyze_text(ticket_text)
label = sentiment_model.get_label(score)

frustration = sentiment_model.compute_frustration(
    sentiment_score=score,
    urgency="medium",
    resolution_time_days=0
)

sentiment = {
    "sentiment_score": round(score, 4),
    "sentiment_label": label,
    "frustration_level": round(frustration, 4)
}

# -------------------------------
# INTENT (FIXED PATH)
# -------------------------------
intent_result = predict(ticket_text, save_dir="../models/nlp")

# -------------------------------
# PRINT OUTPUT
# -------------------------------
print("\n=== NLP OUTPUT ===")
print(f"Sentiment: {sentiment['sentiment_label']} ({sentiment['sentiment_score']:.4f})")
print(f"Frustration Level: {sentiment['frustration_level']:.4f}")

print(f"Intent: {intent_result['intent_group']} ({intent_result['confidence_score']:.4f})")


=== NLP OUTPUT ===
Sentiment: negative (-0.5106)
Frustration Level: 0.7553
Intent: duplicate (0.6196)


## 2. RAG Retrieval

In [25]:
retrieved_raw = retrieve(ticket_text, top_k=5,index_path="../data/retrieval/kb_index")

# ✅ IMPROVED: Show raw retrieval
print("\n=== RAW RETRIEVAL ===")
for r in retrieved_raw[:3]:
    print(f"Sim: {r['similarity_score']:.3f} | {r['solution_comments'][:80]}...")

[FAISS Index] Loaded — 303 vectors from '../data/retrieval/kb_index.faiss'.

=== RAW RETRIEVAL ===
Sim: 0.662 | i can confirm encountering this exact issue on an nvidia geforce rtx 5080 gpu us...
Sim: 0.604 | hi , apologies for the delay, and thanks for raising your concern here. i tried ...
Sim: 0.591 | why there isn't a constant updates through earliest versions on thu, mar 12, 202...


In [26]:
retrieved = optimize_retrieval(
    retrieved_raw,
    intent_label=intent_result["intent_group"],
    intent_confidence=intent_result["confidence_score"]
)

In [27]:
confidence_info = compute_retrieval_confidence(retrieved)

print("\n=== RETRIEVAL CONFIDENCE ===")
print(confidence_info)


=== RETRIEVAL CONFIDENCE ===
{'knowledge_gap_flag': 0, 'retrieval_confidence_level': 'moderate', 'max_sim_seen': 0.6619}


In [28]:
if retrieved:
    knowledge = retrieved[0]["solution_comments"]
    top_sim = retrieved[0]["similarity_score"]
else:
    knowledge = "No solution found."
    top_sim = 0.0

print("\n=== SELECTED KNOWLEDGE ===")
print(f"Top Similarity: {top_sim:.3f}")
print(knowledge[:200])


=== SELECTED KNOWLEDGE ===
Top Similarity: 0.662
i can confirm encountering this exact issue on an nvidia geforce rtx 5080 gpu using tensorflow 2.16+. the issue seems indeed related to the lack of precompiled kernels for compute capability 12.0 (bla


## 3. RL Action Selection

In [ ]:
import torch
from rl.level1_dqn import Level1Agent
from rl.level2_dqn import Level2Agent
from rl.environment import NexResolveEnv
from rl.action_masking import get_action_mask   # ✅ FIX

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------------
# ENV
# -------------------------------
env = NexResolveEnv('../data/final/rl_ready_dataset.csv')
from rl.state_builder import build_single_state

# -------------------------------
# BUILD RL STATE (CORRECT WAY)
# -------------------------------

'''sample_row = {
    "clean_text": ticket_text,

    # ---- Ticket meta ----
    "text_length": len(ticket_text),
    "word_count": len(ticket_text.split()),
    "question_mark_flag": int("?" in ticket_text),
    "has_solution_comment": 0,
    "turn_count": 1,

    # ---- SLA ----
    "sla_breach_flag": 0,
    "sla_remaining_norm": 0.8,
    "sla_limit_hours": 24,
    "interaction_depth": 0,

    # ---- Intent ----
    "intent_group": intent_result["intent_group"],
    "confidence_score": intent_result["confidence_score"],
    "uncertainty_flag": 0,
    "confidence_band": "medium",

    # ---- Urgency ----
    "urgency_score": 0.5,
    "urgent_flag": 0,

    # ---- Entity ----
    "has_version": 0,
    "has_error_type": 1,
    "has_platform": 0,
    "has_hardware": 0,
    "entity_count": 1,
    "missing_count": 2,
    "completeness_score": 0.5,
    "reassignment_count": 0,

    # ---- Clarification ----
    "needs_clarification": 0,
    "clarification_priority": 1,

    # ---- Sentiment ----
    "sentiment_score": sentiment["sentiment_score"],
    "sentiment_label": sentiment["sentiment_label"],
    "frustration_level": sentiment["frustration_level"],

    # ---- Routing ----
    "rl_recommendation": "clarify_first",
    "reopen_count": 0,
    "resolution_success": 0,
    
    "top_tier": "tier1_verified",   # or dynamic if available
    "tier1_flag": 1,
    "tier2_flag": 0,
}'''
high_sim_rows = df[df["max_sim"] >= 0.55]
sample_row = high_sim_rows.iloc[0]


state = build_single_state(sample_row, index_path="../data/retrieval/kb_index")

print("State shape:", state.shape)

# -------------------------------
# L1
# -------------------------------
l1 = Level1Agent(state_dim=state.shape[0])

checkpoint_l1 = torch.load("../models/rl/l1_best.pth", map_location=DEVICE)
l1.online_net.load_state_dict(checkpoint_l1["online_state_dict"])
l1.online_net.eval()

# -------------------------------
# L2
# -------------------------------
l2 = Level2Agent(state_dim=state.shape[0])
l2.load_all("../models/rl/best")

# -------------------------------
# DECISION
# -------------------------------
mask = get_action_mask(state)   # ✅ FIXED

strategy_idx = l1.select_action(state, mask)

from rl.action_space import Strategy

strategy = Strategy(strategy_idx).name.lower()

action = l2.select_action(state, strategy_idx, mask)

print("\n=== RL DECISION ===")
print(f"Strategy: {strategy}")
print(f"Action Index: {action}")

State shape: (37,)

=== RL DECISION ===
Strategy: suggest
Action Index: 13


In [30]:
from rl.action_masking import get_action_mask

In [31]:
mask = get_action_mask(state)

# Level 1 decision
strategy_idx = l1.select_action(state, mask)

# Level 2 decision (FIXED ORDER)
action_idx = l2.select_action(state, strategy_idx, mask)

print("Strategy Index:", strategy_idx)
print("Action Index:", action_idx)

Strategy Index: 2
Action Index: 13


In [32]:
print("\n=== RL DECISION ===")

# Rule baseline (for explanation)
if confidence_info["knowledge_gap_flag"] == 1:
    rule_strategy = "CLARIFY"
elif confidence_info["max_sim_seen"] >= 0.55:
    rule_strategy = "SUGGEST"
else:
    rule_strategy = "ESCALATE"

print("Rule-based strategy:", rule_strategy)

# DQN decision
print("DQN strategy index:", strategy_idx)
print("DQN action index:", action_idx)


=== RL DECISION ===
Rule-based strategy: SUGGEST
DQN strategy index: 2
DQN action index: 13


## 4. Final Response Generation

In [33]:
# -------------------------------
# SIMILARITY GUARD
# -------------------------------
top_sim = retrieved[0]["similarity_score"] if retrieved else 0

print(f"\nTop Similarity: {top_sim:.3f}")
print(f"Chosen Strategy: {strategy}")

# Only generate if meaningful retrieval AND correct strategy
if top_sim >= 0.50 and strategy == "suggest":
    
    print("\nGenerating response...\n")

    generator = ResponseGenerator(model_type="FLAN")  # ✅ removed device="cpu"

    response = generator.generate(
        ticket_summary=ticket_text,
        intent=intent_result["intent_group"],
        entities={},
        retrieved_solution=knowledge
    )

    print("\n=== FINAL RESPONSE ===")
    print(response)

else:
    print("\nSkipping generation:")
    
    if top_sim < 0.50:
        print("Reason: Knowledge gap too high (low similarity)")
    
    if strategy != "suggest":
        print(f"Reason: Strategy is '{strategy}', not 'suggest'")


Top Similarity: 0.662
Chosen Strategy: suggest

Generating response...

Loading FLAN model from google/flan-t5-large onto cuda...

=== FINAL RESPONSE ===
i can confirm encountering this exact issue on an nvidia geforce rtx 5080 gpu using tensorflow 2.16+. the issue seems indeed related to the lack of precompiled kernels for compute capability 12.
